In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
DATA_PATH = Path("../data/train.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(57477, 9)


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57477 entries, 0 to 57476
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              57477 non-null  int64 
 1   model_a         57477 non-null  object
 2   model_b         57477 non-null  object
 3   prompt          57477 non-null  object
 4   response_a      57477 non-null  object
 5   response_b      57477 non-null  object
 6   winner_model_a  57477 non-null  int64 
 7   winner_model_b  57477 non-null  int64 
 8   winner_tie      57477 non-null  int64 
dtypes: int64(4), object(5)
memory usage: 3.9+ MB


no NA

In [4]:
label_cols = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie"
]

df[label_cols].sum()

winner_model_a    20064
winner_model_b    19652
winner_tie        17761
dtype: int64

In [5]:
df[label_cols].mean()

winner_model_a    0.349079
winner_model_b    0.341911
winner_tie        0.309011
dtype: float64

In [6]:
df[label_cols].sum(axis=1).value_counts() #each line has one and the only label (A/B/tie)

1    57477
dtype: int64

In [7]:
#merge the 3 one-hot code into 1 label (A/B/tie)
df["label"] = np.select(
    [
        df["winner_model_a"] == 1,
        df["winner_model_b"] == 1,
        df["winner_tie"] == 1
    ],
    [
        "A",
        "B",
        "tie"
    ]
)

In [8]:
df["label"].value_counts()

A      20064
B      19652
tie    17761
Name: label, dtype: int64

In [9]:
type(df.loc[0, "prompt"]) #the type of prompt is str rather than list!

str

In [13]:
sample_prompt = df.loc[0, "prompt"]

parsed_prompt = json.loads(sample_prompt) #transfer the str to a real list
parsed_a = json.loads(df.loc[0, "response_a"])
parsed_b = json.loads(df.loc[0, "response_b"])

print(type(parsed_prompt))
print(parsed_prompt)
print(len(parsed_prompt), len(parsed_a), len(parsed_b))

<class 'list'>
['Is it morally right to try to have a certain percentage of females on managerial positions?', 'OK, does pineapple belong on a pizza? Relax and give me fun answer.']
2 2 2


In [14]:
for i, x in enumerate(df["prompt"]):
    parsed = json.loads(x)
    if len(parsed) > 1:
        print("row index:", i)
        print("turns:", len(parsed))
        print(parsed)
        break

row index: 0
turns: 2
['Is it morally right to try to have a certain percentage of females on managerial positions?', 'OK, does pineapple belong on a pizza? Relax and give me fun answer.']


In [15]:
df["prompt_list"] = df["prompt"].apply(json.loads)
df["response_a_list"] = df["response_a"].apply(json.loads)
df["response_b_list"] = df["response_b"].apply(json.loads)

In [16]:
df["prompt_turns"] = df["prompt_list"].apply(len)
df["a_turns"] = df["response_a_list"].apply(len)
df["b_turns"] = df["response_b_list"].apply(len)

In [17]:
df[["prompt_turns", "a_turns", "b_turns"]].describe()

,prompt_turns,a_turns,b_turns
count,57477.000000,57477.000000,57477.000000
mean,1.244219,1.244219,1.244219
std,0.908211,0.908211,0.908211
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,36.000000,36.000000,36.000000


In [20]:
df["prompt_turns"].value_counts().sort_index()

1     49938
2      4673
3      1485
4       607
5       311
6       167
7       103
8        60
9        44
10       29
11       16
12       14
13        5
14        3
15        4
16        2
17        3
18        1
19        4
20        2
21        1
22        1
29        1
32        1
35        1
36        1
Name: prompt_turns, dtype: int64

In [21]:
multi_turn_ratio = (df["prompt_turns"] > 1).mean()

print(f"{multi_turn_ratio:.2%}")

Multi-turn conversation ratio: 13.12%


In [22]:
#check whether the turns of the prompt and response are consistent
prompt_a_match = (df["prompt_turns"] == df["a_turns"]).mean()
prompt_b_match = (df["prompt_turns"] == df["b_turns"]).mean()

print(f"{prompt_a_match:.2%}")
print(f"{prompt_b_match:.2%}")

Prompt-A turn match: 100.00%
Prompt-B turn match: 100.00%
